In [0]:

-- Cluster-aware season overview
CREATE OR REFRESH MATERIALIZED VIEW mart_season_overview AS
WITH last_week AS (
  SELECT league_id, season, roster_id, max(week) AS max_week
  FROM workspace.sleeper_core.fact_standings_week
  GROUP BY league_id, season, roster_id
),
asof AS (
  SELECT s.*
  FROM workspace.sleeper_core.fact_standings_week s
  JOIN last_week m
    ON s.league_id=m.league_id AND s.season=m.season AND s.roster_id=m.roster_id AND s.week=m.max_week
),
clusters AS (
  SELECT league_id,
         lower(regexp_replace(coalesce(name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
         name AS cluster_name
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
)
SELECT c.cluster_key, c.cluster_name,
       a.league_id, a.season, a.roster_id, a.wins, a.losses, a.ties,
       a.points_for, a.points_against, a.expected_wins,
       cm.points_for_stddev, cm.points_for_cv, cm.games_played
FROM asof a
LEFT JOIN workspace.sleeper_core.agg_consistency_metrics cm
  ON a.league_id=cm.league_id AND a.season=cm.season AND a.roster_id=cm.roster_id
LEFT JOIN clusters c USING (league_id);

-- Trade leaderboards (unchanged shape, add cluster as a separate view)
CREATE OR REFRESH MATERIALIZED VIEW mart_trade_leaderboard AS
SELECT league_id, side_roster_id, total_points, total_faab, total_impact
FROM workspace.sleeper_trades.agg_trade_impact_all_time
ORDER BY total_impact DESC;

CREATE VIEW mart_trade_leaderboard_cluster AS
SELECT
  lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  li.name AS cluster_name,
  l.*
FROM mart_trade_leaderboard l
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id);

CREATE OR REFRESH MATERIALIZED VIEW mart_trade_summary AS
SELECT league_id, season, transaction_id, side_roster_id,
       sum_post_points, sum_pick_points, sum_faab_delta, season_impact_total,
       row_number() OVER (PARTITION BY league_id, season ORDER BY season_impact_total DESC) AS rank_in_league_season
FROM workspace.sleeper_trades.agg_trade_impact_per_season;

CREATE VIEW mart_trade_summary_cluster AS
SELECT
  lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  li.name AS cluster_name,
  s.*
FROM mart_trade_summary s
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id);

CREATE OR REFRESH MATERIALIZED VIEW mart_player_performance AS
SELECT f.league_id, f.season, f.week, f.roster_id, f.player_id,
       p.full_name, p.position, f.points, f.projected_points
FROM workspace.sleeper_core.fact_player_week f
LEFT JOIN workspace.sleeper_core.dim_players p
  ON f.player_id = p.player_id;

CREATE VIEW mart_player_performance_cluster AS
SELECT
  lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  li.name AS cluster_name,
  m.*
FROM mart_player_performance m
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id);

CREATE OR REFRESH MATERIALIZED VIEW mart_waiver_activity AS
SELECT * FROM workspace.sleeper_core.evt_waiver_events;

CREATE VIEW mart_waiver_activity_cluster AS
SELECT
  lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  li.name AS cluster_name,
  w.*
FROM mart_waiver_activity w
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id);

CREATE OR REFRESH MATERIALIZED VIEW mart_draft_value_roi AS
SELECT
  r.league_id,
  r.season,
  r.round,
  r.avg_points,
  r.season_avg_points,
  r.avg_roi,
  d.value
FROM sleeper_core.agg_draft_roi_by_round r
LEFT JOIN sleeper_core.dim_draft_pick_value d
  ON r.season = d.season AND r.round = d.round;

CREATE VIEW mart_draft_value_roi_cluster AS
SELECT
  lower(regexp_replace(coalesce(li.name,'unknown'), '[^a-zA-Z0-9]+', '_')) AS cluster_key,
  li.name AS cluster_name,
  v.*
FROM mart_draft_value_roi v
JOIN workspace.sleeper_raw.sleeper_league_info_snapshot li USING (league_id);
